<a href="https://colab.research.google.com/github/tanishka-jadhav/NLP-Preprocessing-Engine/blob/main/ASSIGNMENT_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ASSIGNMENT-4

NAME: Tanishka Prasad Jadhav
 ID: IN226010302

NLP Assignment 4 — Fine-Tuning BERT


In [ ]:
#1. Install Libraries
!pip install transformers
!pip install torch
!pip install scikit-learn
!pip install pandas

In [ ]:
#Import Libraries
import torch
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

import os
os.environ["DISABLE_WIDGETS"] = "1"
from IPython.display import display

In [ ]:
#3. Load Dataset (IMDB Sample – No Kaggle Needed)
# Create simple dataset (safe + fast)
data_dict = {
    "sentence": [
        "This movie was fantastic",
        "I hated the film",
        "It was an excellent performance",
        "Very boring storyline",
        "Loved the cinematography",
        "Not worth watching",
        "Absolutely amazing",
        "Worst movie ever",
        "Really enjoyed it",
        "Terrible acting"
    ],
    "target": [1,0,1,0,1,0,1,0,1,0]
}

dataset = pd.DataFrame(data_dict)

In [ ]:
#4. Train-Test Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

In [ ]:
#5. Tokenization (BERT)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

In [ ]:
#6. Create Dataset Class
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
test_dataset = Dataset(test_encodings, test_labels)

In [ ]:
#7. Load BERT Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
#8. Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_dir="./logs",
    logging_steps=10
)

In [ ]:
#9. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [ ]:
#10. Train Model
trainer.train()
# 11. Predictions
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)

In [ ]:
#12. Evaluation Metrics
accuracy = accuracy_score(test_labels, preds)
precision = precision_score(test_labels, preds)
recall = recall_score(test_labels, preds)
f1 = f1_score(test_labels, preds)
cm = confusion_matrix(test_labels, preds)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)
'''Accuracy: 0.0
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Confusion Matrix:
 [[0 1]
 [1 0]]'''